In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor


def load_and_split(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values

def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values


In [2]:
# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
print(X.shape)
X = X[:300000, :]
y = y[:300000]

def load_and_split_2(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    X_hp, X_val, y_hp, y_val = train_test_split(
        y_val, y_val, test_size=0.5, random_state=random_state
    )
    return X_train, X_val, X_hp, X_test, y_train, y_val, y_hp, y_test

name = 'facebook_comment_volume'
X_train, X_val, X_hp, X_test, y_train, y_val, y_hp, y_test = load_and_split_2(X, y)
del X, y

(583250, 77)


In [ ]:
from catboost import CatBoostRegressor
import csv
import os

import sys
sys.path.append('..')

from src.model_generator import CopyModelGenerator, OptunaModelGenerator
from src.gradient_boosting_regressor import MyCatBoost
import optuna


study = optuna.create_study(
    direction="minimize",  # RMSE
    sampler=optuna.samplers.TPESampler()
)


single_tree_model = CatBoostRegressor(
    iterations=1,
    learning_rate=1.0,
    loss_function='RMSE',
    verbose=False
)
model_generator = OptunaModelGenerator(single_tree_model, study)
gbrt = MyCatBoost(
    model_generator=model_generator,
    n_estimators=2000,
    learning_rate=0.1,
    verbose=1
)

gbrt.fit(
    X_train, y_train,
    #hp_search_set=(X_hp, y_hp),
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
)

pred = gbrt.predict(X_test)
r2 = r2_score(y_test, pred)
r2_val = r2_score(y_val, gbrt.predict(X_val))

[I 2026-01-18 20:02:04,927] A new study created in memory with name: no-name-0c8b0625-69dc-4b0d-963b-80d15c3e423d


Pre-algo time: 0.0009829998016357422 s
[0] train RMSE=619.716802, val RMSE=560.101034
[1] train RMSE=563.467740, val RMSE=560.855633
[2] train RMSE=536.674296, val RMSE=560.365027
[3] train RMSE=502.446240, val RMSE=559.987716
[4] train RMSE=457.421857, val RMSE=563.451744
[5] train RMSE=424.022182, val RMSE=563.482938
[6] train RMSE=388.762060, val RMSE=566.996510
[7] train RMSE=355.770116, val RMSE=567.031120
[8] train RMSE=335.671904, val RMSE=565.840762
[9] train RMSE=322.439487, val RMSE=570.607729
[10] train RMSE=317.582345, val RMSE=575.983439
[11] train RMSE=313.592860, val RMSE=581.681247
[12] train RMSE=309.714955, val RMSE=590.129528
[13] train RMSE=303.401151, val RMSE=592.309458
[14] train RMSE=294.518336, val RMSE=590.474302
[15] train RMSE=284.701407, val RMSE=595.286238
[16] train RMSE=276.584565, val RMSE=601.023501
[17] train RMSE=268.818146, val RMSE=607.109254
[18] train RMSE=258.941164, val RMSE=607.141235
[19] train RMSE=249.865627, val RMSE=607.170027
[20] train 

KeyboardInterrupt: 

In [ ]:
print(r2)

0.09209261782261502
